# Pipeline completo no Colab gratuito (Nível 1)

Reproduz a base, o ranking, a categorização, a análise, o playbook e o agente. Roda em CPU; o Nível 2 (transcrição) fica mais rápido com GPU T4.

## 1. Código
Envie o zip do repositório (menu Arquivos) **ou** troque a URL abaixo pela do seu repositório.

In [ ]:
import os
REPO_URL = ''  # ex.: https://github.com/<usuario>/stone-engajamento-mpe.git
if REPO_URL:
    !git clone -q $REPO_URL projeto
else:
    !unzip -q -o /content/*.zip -d projeto
%cd /content/projeto
!pip install -q -e .
!apt-get -qq install -y ffmpeg > /dev/null

## 2. Chaves
Cadastre `YOUTUBE_API_KEY`, `GEMINI_API_KEY` e `APIFY_TOKEN` (Instagram) em **Secrets** (ícone de chave na barra lateral) e libere o acesso ao notebook.

In [ ]:
from google.colab import userdata
for k in ['YOUTUBE_API_KEY', 'GEMINI_API_KEY', 'OPENROUTER_API_KEY', 'APIFY_TOKEN']:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('sem', k)
os.environ['LLM_PROVIDERS'] = 'gemini,openrouter,none'

## 3. Dados do Instagram (rota do kickoff)
Se tiver o export, envie e importe. Senão, pule: a base fica só com YouTube e isso é declarado no README.

In [ ]:
# Instagram via Apify (rota do kickoff): estima o custo antes (~US$ 2,20 no crédito gratuito)
!python -m stone_mpe.cli collect-instagram --dry-run
# !python -m stone_mpe.cli collect-instagram

## 4. Pipeline
Coleta YouTube → base → categorização do top → amostra do resto (lift) → comentários → análise → playbook → cache da demo. Re-executar continua do checkpoint.

In [ ]:
!python -m stone_mpe.cli pipeline

## 5. Validação manual (≥ 50)
Gere a amostra, baixe `validation/manual_sample.csv`, rotule **sem olhar a saída do modelo**, envie de volta e calcule a concordância.

In [ ]:
!python -m stone_mpe.cli sample -n 50
# depois de rotular:
# !python -m stone_mpe.cli agreement

## 6. Agente

In [ ]:
from stone_mpe.api import run
out = run('gerar_criativo', 'Preciso de um Reels para atrair donos de pequenos restaurantes', segment='alimentacao')
print(out.explanation)
for c in out.creatives:
    print('
GANCHO:', c.gancho, '
CTA:', c.cta, '
REFERÊNCIAS:', c.inspired_by)
for r in out.reference_posts:
    print(r.handle, r.url, f'{r.engagement_rate:.2%}')

## 7. Playbook

In [ ]:
from IPython.display import Markdown
Markdown(open('outputs/playbook.md', encoding='utf-8').read())